In [0]:
from datetime import datetime

In [0]:
file_path = "/Volumes/dev_catalog/landing/landing_vol/raw_roads/"
file_path2 = "/Volumes/dev_catalog/landing/landing_vol/raw_traffic/"
notebook_name = "02_load_to_bronze_batch"

In [0]:
def log_pipeline_error(step_name, error):
    spark.createDataFrame(
        [(notebook_name, step_name, str(error), datetime.now())],
        ["notebook", "step", "error_message", "error_time"]
    ).write.mode("append").saveAsTable("dev_catalog.default.pipeline_errors")

In [0]:
%sql DESCRIBE EXTENDED dev_catalog.default.pipeline_errors

In [0]:
try:
    df = spark.read.csv(file_path, header=True, inferSchema=True)
except Exception as e:
    log_pipeline_error("read_csv_road", e)
    raise


In [0]:

try:
    spark.sql("""
CREATE OR REPLACE TABLE dev_catalog.bronze.raw_roads
LOCATION 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/raw_roads'
AS
SELECT *, current_timestamp() AS ingestion_timestamp
from read_files('/Volumes/dev_catalog/landing/landing_vol/raw_roads/', 
    format =>'csv',
    header => true,
    inferSchema => true
    
);
  """ )
except Exception as e:
    log_pipeline_error("create_table_road",e)
    raise

In [0]:
%sql
SELECT current_user();

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION bronze;

In [0]:
try:
    df = spark.read.csv(path=file_path2, header=True, inferSchema=True)
    df.display()
    df.schema
except Exception as e:
    log_pipeline_error("read_csv_traffic", e)
    raise

In [0]:
try:
    spark.sql("""CREATE TABLE IF NOT EXISTS dev_catalog.bronze.raw_traffic(
        count_point_id STRING, year INT, region_id INT, region_name STRING, region_ons_code STRING, local_authority_id DOUBLE, local_authority_name STRING, local_authority_code STRING, road_name STRING, road_category STRING, road_type STRING, start_junction_road_name STRING, end_junction_road_name STRING, easting INT, northing INT, latitude DOUBLE, longitude DOUBLE, link_length_km STRING, link_length_miles STRING, estimation_method STRING, estimation_method_detailed STRING, direction_of_travel STRING, pedal_cycles INT, two_wheeled_motor_vehicles INT, cars_and_taxis INT, buses_and_coaches INT, LGVs INT, HGVs_2_rigid_axle INT, HGVs_3_rigid_axle INT, HGVs_4_or_more_rigid_axle INT, HGVs_3_or_4_articulated_axle INT, HGVs_5_articulated_axle INT, HGVs_6_articulated_axle INT, all_HGVs INT, all_motor_vehicles INT
    )
    LOCATION 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/raw_traffic'
    """)
except Exception as e:
    log_pipeline_error("create_table_raw_traffic",e)
    raise

In [0]:
try:
    spark.sql("""
            COPY INTO dev_catalog.bronze.raw_traffic
            FROM '/Volumes/dev_catalog/landing/landing_vol/raw_traffic/'
            FILEFORMAT = CSV
            FORMAT_OPTIONS('header' = 'true' , 'inferSchema' = 'true')
            COPY_OPTIONS ('mergeSchema' = 'true');
        """)
except Exception as e:
    log_pipeline_error("create_table_raw_traffic",e)
    raise

In [0]:
try:
    spark.sql("""
           CREATE OR REPLACE TABLE dev_catalog.bronze.vehicle_type_lookup (
            vehicle_code STRING,
            vehicle_label STRING
            );""")
    
    spark.sql("""
            INSERT INTO dev_catalog.bronze.vehicle_type_lookup VALUES
            ('EV_Car', 'Electric Car'), ('EV_Bike', 'Electric Bike'),
            ('LGV_Type', 'Light Goods Vehicle'), ('HGV_Type', 'Heavy Goods Vehicle')
        """)
except Exception as e:
    log_pipeline_error("create_table_raw_traffic",e)
    raise

In [0]:
%sql
SELECT * from dev_catalog.default.pipeline_errors;